> **Hands-on workshop for the AI-Driven DevOps (AIDD) program, School of Computer Science and Engineering, Jain (Deemed-to-be University), Bengaluru**

# 05 · AgentCore Memory: remembering students

**CampusMate v2 (an agent that remembers)**  ·  ⏱️ Time budget: **15 min theory + 30 min hands-on**

*All student, course, policy and event data in this workshop is fictional.*

## Why this matters

A good mentor remembers you're a 3rd-semester student who loves cloud computing, without you repeating it every time.
In notebook 04, a new `Agent(...)` forgets everything. **AgentCore Memory** stores conversations in AWS and pulls out lasting facts,
so CampusMate still knows you next week. Today the agent runs on your laptop; in notebook 07 the same agent moves to the cloud.

### What you'll do
- Learn short-term vs long-term memory
- Create your own AgentCore Memory in AWS (one cell)
- Give your notebook-04 CampusMate that memory
- Tell it a fact, start a NEW conversation, and see it remember

## Settings

✏️ Change `STUDENT_ID` to your own ID before running anything else (lower-case letters and digits, e.g. `priya07`). Use the same ID in notebooks 05-07.

In [ ]:
# ---- Step 0: make sure the workshop libraries are in THIS notebook's Python (macOS, Windows, Linux) ----
import sys, subprocess, importlib, importlib.util
if sys.version_info < (3, 10):
    raise SystemExit(f"This notebook runs Python {sys.version.split()[0]}. Choose the .venv kernel (Python 3.10+) and run again.")
if any(importlib.util.find_spec(m) is None for m in ("strands", "strands_tools", "bedrock_agentcore", "mcp_proxy_for_aws", "boto3", "dotenv")):
    print("Installing the workshop libraries into", sys.executable, "(1-2 minutes, only the first time)...")
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
                               "strands-agents==1.64.0", "strands-agents-tools==0.9.0", "bedrock-agentcore==1.25.4",
                               "mcp-proxy-for-aws==1.8.0", "boto3==1.43.126", "botocore==1.43.126",
                               "flask==3.1.3", "python-dotenv==1.2.4", "requests==2.34.2"])
        importlib.invalidate_caches()
    except subprocess.CalledProcessError:
        raise SystemExit("Could not install here. Choose the .venv kernel (VS Code: Select Kernel -> .venv; "
                         "Jupyter: start it from a terminal after activating .venv), then run this cell again.")
print("Libraries ready in:", sys.executable)

# Settings for this lab
import json, time, uuid, pathlib
import boto3

# ✏️ Put YOUR student ID here (the same one in notebooks 05-07). Lower-case letters and digits, e.g. priya07
STUDENT_ID = "student01"

REGION = "us-east-1"
MODEL_ID = "us.amazon.nova-2-lite-v1:0"              # Amazon Nova 2 Lite
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
DATA_DIR = ROOT / "data"
MEMORY_NAME = f"campusmate_mem_{STUDENT_ID}"           # your Memory in AWS
print("Student:", STUDENT_ID, "| Memory name:", MEMORY_NAME)

## Two kinds of memory

| | Short-term memory | Long-term memory |
|---|---|---|
| What | Every message (an **event**) in a session | **Records** pulled out of the events |
| When | Saved instantly | Pulled out **in the background**, after about 30-90 seconds |
| Scope | One session | All sessions of one student (the **actor**) |
| Campus analogy | Today's attendance register | Your file in the department office |

## Two strategies decide what goes into long-term memory

- **SEMANTIC**: facts and preferences, e.g. *"Student is in 3rd semester; interested in cloud."* Stored under `/users/<student>/facts`.
- **SUMMARIZATION**: a short recap of each session. Stored under `/summaries/<student>/<session>`.

Our **actor ID is the student ID** (`AIDD-DEMO-001`), so each student's memories stay separate. The **session ID** is one conversation.

## Step 1: create your Memory in AWS

One Memory per workshop student, with the two strategies above. The first time it takes **2-3 minutes**; if it already exists, the cell just reuses it.

In [ ]:
from bedrock_agentcore.memory import MemoryClient

memory_client = MemoryClient(region_name=REGION)
existing = [m for m in memory_client.list_memories() if m.get("id", "").startswith(MEMORY_NAME + "-")]
if existing:
    MEMORY_ID = existing[0]["id"]
    print("Reusing your Memory:", MEMORY_ID)
else:
    print("Creating your Memory (2-3 minutes)...")
    memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="CampusMate workshop memory",
        strategies=[
            {"semanticMemoryStrategy": {"name": "facts", "namespaces": ["/users/{actorId}/facts"]}},
            {"summaryMemoryStrategy": {"name": "summaries", "namespaces": ["/summaries/{actorId}/{sessionId}"]}},
        ],
        event_expiry_days=30,
    )
    MEMORY_ID = memory["id"]
    print("Created:", MEMORY_ID)

## Step 2: your CampusMate, now with memory

The same two tools from notebook 04. The only new part is `session_manager=...`:
- it **saves** every message to your Memory (short-term), and
- before each answer, it **brings back** what is known about this student (long-term).

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfig
from bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManager


@tool
def get_course_info(query: str) -> str:
    """Look up AIDD courses by course code (e.g. AIDD303) or a keyword in the title/description (e.g. docker).

    Args:
        query: A course code or a keyword.
    """
    q = query.lower()
    courses = json.loads((DATA_DIR / "courses.json").read_text(encoding="utf-8"))["courses"]
    hits = [c for c in courses if q == c["code"].lower() or q in (c["title"] + " " + c["description"]).lower()]
    return json.dumps(hits[:5]) if hits else json.dumps({"found": False, "codes": [c["code"] for c in courses]})


@tool
def get_campus_policy(topic: str) -> str:
    """Return CampusMate rules for a topic: attendance, internal_assessments, semester_end_exams, fees, leave,
    re_evaluation, library, hackathons. Always use this instead of guessing a rule.

    Args:
        topic: one of the topics above
    """
    policies = json.loads((DATA_DIR / "campus_policies.json").read_text(encoding="utf-8"))["policies"]
    return json.dumps(policies.get(topic.lower().strip(), {"found": False, "topics": list(policies)}))


def campusmate_with_memory(session_id, student):
    """A CampusMate that saves this conversation and remembers what it knows about this student."""
    memory = AgentCoreMemorySessionManager(
        AgentCoreMemoryConfig(memory_id=MEMORY_ID, session_id=session_id, actor_id=student,
                              retrieval_config={"/users/{actorId}/facts": RetrievalConfig(top_k=5),
                                                "/summaries/{actorId}": RetrievalConfig(top_k=3)}),
        region_name=REGION)
    return Agent(
        model=BedrockModel(model_id=MODEL_ID, max_tokens=1200),
        tools=[get_course_info, get_campus_policy],
        session_manager=memory,
        system_prompt=f"You are CampusMate for AIDD students, talking to student {student}. "
                      "Use tools for courses and rules; never invent rules. Be brief.",
        callback_handler=None,
    )

print("CampusMate with memory is ready.")

## Step 3: tell CampusMate something about yourself

We pretend to be student **AIDD-DEMO-001**, in conversation (session) **A**.

In [ ]:
STUDENT = "AIDD-DEMO-001"
session_a = f"lab05-A-{uuid.uuid4()}"
agent_a = campusmate_with_memory(session_a, STUDENT)
print(agent_a("Hi! I'm in 3rd semester and I'm really interested in cloud computing."))

## Step 4: wait for long-term memory (it works in the background)

Pulling facts out of a conversation happens **in the background**, usually within 30-90 seconds. If we ask a new session too early,
it won't know anything yet. So we check every 15 seconds until the fact appears. This waiting is normal, not a bug.

In [ ]:
def facts_about(student):
    records = memory_client.retrieve_memories(memory_id=MEMORY_ID, namespace=f"/users/{student}/facts",
                                              query="semester and interests", top_k=5)
    return [r["content"]["text"] for r in records]

for attempt in range(12):            # up to about 3 minutes
    found = facts_about(STUDENT)
    if any("cloud" in f.lower() for f in found):
        break
    print(f"  attempt {attempt + 1}: not there yet, waiting 15 s...")
    time.sleep(15)
print("Long-term facts:", found)

## 🔬 Live check: a brand-new conversation remembers

New session **B**, same student. This is a fresh `Agent(...)`: without Memory it would know nothing (remember notebook 01).

In [ ]:
session_b = f"lab05-B-{uuid.uuid4()}"     # NEW conversation, same student
agent_b = campusmate_with_memory(session_b, STUDENT)
print(agent_b("Suggest one topic I should explore next. Use what you know about me."))

## ✍️ Your turn

Teach CampusMate a **preference** as a different student (for example *"Please always answer me in bullet points"*),
wait for it to appear in memory, then check that a new conversation follows it.

> The cell below runs safely as-is (placeholders are skipped). Replace each `# TODO` and run it again.

In [ ]:
# YOUR TURN
OTHER_STUDENT = "AIDD-DEMO-008"
MY_PREFERENCE = "TODO"     # TODO: e.g. "Please always answer me in bullet points."

if MY_PREFERENCE != "TODO":
    print(campusmate_with_memory(f"pref-{uuid.uuid4()}", OTHER_STUDENT)(MY_PREFERENCE))
    # TODO: wait using facts_about(OTHER_STUDENT) like Step 4, then ask a question in a new session
else:
    print("Skipped: set MY_PREFERENCE first.")

## ✅ Checkpoint

You are on track if:
- Step 1 printed `Created:` or `Reusing your Memory:` with an ID
- Step 4 printed a fact like *in 3rd semester* / *interested in cloud computing*
- The new conversation's suggestion mentions cloud

Your Memory stays in AWS: notebook 07 uses AgentCore again, and its last cell cleans everything up.

## 🛠️ Common errors

| You see | Why | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'strands'` | The notebook isn't using `.venv` | Run the Settings cell again, or VS Code: Select Kernel → `.venv` |
| `AccessDeniedException ... bedrock-agentcore` | Your IAM user lacks AgentCore permissions | Use the admin IAM user from the setup steps |
| Facts list stays empty | Long-term memory works in the background and can take a few minutes | Wait and re-run the Step 4 cell |
| The new conversation doesn't remember | The fact wasn't extracted yet, or a different student ID was used | Wait for Step 4, use the same `STUDENT` |
| `ThrottlingException` | Many students at once | Wait 30 seconds and run the cell again |